In [ ]:
import importlib.util
import subprocess
import sys

from pathlib import Path
import pandas as pd

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from config import SEED, DATA_DIR, CATEGORIES, REVIEWS_PER_SUMMARY, MAX_WORDS_PER_REVIEW, MODEL_NAME, SUMMARY_INSTRUCTIONS, MAX_SUMMARY_LENGTH, SUMMARIES_PATH, TOP_N_TOTAL

In [ ]:
missing = []
for package in ["transformers", "accelerate", "bitsandbytes"]:
    if importlib.util.find_spec(package) is None:
        missing.append(package)

if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--user", "-q", *missing], check=True)
    print(f"Installed {missing}, so restart the kernel and run from the top")

else:
    print("All libraries ready")

In [ ]:
# Loads the uploaded top product reviews and picks the same 50 random 4 and 5 star reviews per product
category_reviews = []

for c in CATEGORIES:
    category_reviews.append(pd.read_parquet(DATA_DIR / f"top_reviews_{c.lower()}_{TOP_N_TOTAL // len(CATEGORIES)}.parquet"))
    
top_reviews = pd.concat(category_reviews, ignore_index=True)

positive_reviews = top_reviews[top_reviews["rating"] >= 4]
base_sets = (positive_reviews.groupby("parent_asin", group_keys=False).sample(n=REVIEWS_PER_SUMMARY, random_state=SEED).reset_index(drop=True))
base_sets.groupby("product_name")["rating"].agg(n_reviews="size", avg_rating="mean").round(2)

In [ ]:
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 2**30

if torch.cuda.get_device_capability()[0] >= 8:
    dtype = torch.bfloat16

else:
    dtype = torch.float16

if gpu_gb < 24:
    PRECISION = "4bit"
    quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=dtype)
else:
    PRECISION = str(dtype).replace("torch.", "")
    quantization_config = None

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype, device_map="cuda", quantization_config=quantization_config)
print(f"Loaded {MODEL_NAME} in {PRECISION} on a {gpu_gb:.0f} GB GPU")

def format_reviews(review_set):
    lines = []

    for number, review in enumerate(review_set.itertuples(), start=1):
        words = str(review.text).split()
        text = " ".join(words[:MAX_WORDS_PER_REVIEW])

        if len(words) > MAX_WORDS_PER_REVIEW:
            text = text + " ..."

        lines.append(f"Review {number} ({int(review.rating)} stars): {review.review_title}. {text}")

    return "\n".join(lines)

def summarize(product_name, review_set):
    content = f"Product: {product_name}\n\nCustomer reviews:\n{format_reviews(review_set)}\n\n{SUMMARY_INSTRUCTIONS}"
    inputs = tokenizer.apply_chat_template([{"role": "user", "content": content}], add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
    
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=MAX_SUMMARY_LENGTH, do_sample=False, temperature=None, top_p=None, top_k=None)
        
    return tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
if SUMMARIES_PATH.exists():
    base_summaries = pd.read_parquet(SUMMARIES_PATH)

else:
    base_summaries = pd.DataFrame()

done = set()
if len(base_summaries) > 0:
    done = set(base_summaries["parent_asin"])

for (asin, name), review_set in base_sets.groupby(["parent_asin", "product_name"]):
    if asin in done:
        continue

    print(f"Summarizing {name[:60]}...")
    row = {
        "parent_asin": asin, 
        "product_name": name, 
        "n_reviews": len(review_set), 
        "model": MODEL_NAME, 
        "precision": PRECISION, 
        "prompt": SUMMARY_INSTRUCTIONS, 
        "summary": summarize(name, review_set)
        }
        
    base_summaries = pd.concat([base_summaries, pd.DataFrame([row])], ignore_index=True)
    base_summaries.to_parquet(SUMMARIES_PATH, index=False)

base_summaries[["product_name", "summary"]]